# Case Study : Bayesian logistic regression via SOUL

This notebook includes my implementation of the SOUL algorithm and its result comparison to the Bayesian logistic regression example in Section 3.1 of [Particle algorithms for maximum likelihood training of latent variable models](https://juankuntz.github.io/publication/parem/).  

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

#@title Load modules.

# Install the wget package on Colab (if running the notebook locally,
# comment the following line out).
!pip install wget

# OS and wget to load dataset.
import os
import wget
from scipy.stats import norm


We load the same dataset in order to compare results and verify the correctness of our function.

In [ ]:
#@title Load and normalize the Wisconsin Breast Cancer dataset.

# Fetch dataset from repository:
wget.download('https://archive.ics.uci.edu/ml/machine-learning-databases/breast-cancer-wisconsin/breast-cancer-wisconsin.data')

# Load dataset:
dataset = np.loadtxt('breast-cancer-wisconsin.data', dtype=str, delimiter=',')

# Delete local copy of dataset to avoid duplicates with multiple notebook runs:
os.remove('breast-cancer-wisconsin.data')

# Remove datapoints with missing attributes and change dtype to float:
dataset = dataset[~(dataset == '?').any(axis=1), :].astype(float)

# Extract features and labels, and normalize features:
features = np.array(dataset[:, 1:10] - dataset[:, 1:10].mean(0))
features = features/features.std(0)
labels = np.array([(dataset[:, 10]-2)/2]).transpose()

# SOUL function.

In [ ]:
#@title SOUL function
# This is my own SOUL function
def soul(log_p_grad_x, th0, x0_M, y_l, y_f, T, M, B, delta_step, gamma_step):
  """
    Stochastic Optimisation via Unadjusted Langevin (SOUL).

    Parameters:
    - log_p_grad_x: Function returning grad_x log p(th, X, y). Shape matches X.
    - log_p_grad_th: Function returning grad_theta log p(th, X, y). Shape matches theta.
    - th_0: Initial parameters
    - x0_M: Initial latent variables from the previous step
    - y_l, y_f: Observed data
    - T: Number of outer optimization steps
    - M: Number of Langevin steps
    - B: Burn-in steps
    - delta_step: Step size for theta update
    - gamma_step: Langevin step size
    """
  th = np.copy(th0)
  x_t = np.copy(x0_M[:, 0:1]).reshape(D,1) # shape (D,1) (instead of (D,)) - keeps the column dimension

  x_values = np.array(x0_M)
  th_list = [th0]
  for t in range(1, T+1):

    # ULA steps
    for m in range(1, M+1):
      grad_x = log_p_grad_x(th, x_t, y_l, y_f)
      Z_k = np.random.normal(0.0, 1.0, x_t.shape) # Force size

      # Compute new x
      x_t += gamma_step * grad_x + np.sqrt(2.0 * gamma_step) * Z_k
      x_values = np.append(x_values, np.copy(x_t), axis=1) # Store new position

    burnin_x_samples = x_values[:, -(M-B):] # Shape (D, M-B)

    # Compute new theta
    avg_grad_th = np.zeros_like(th) # th is (1,1), so avg_grad_th is (1,1)

    # Iterate over particles (columns) instead of feature dimensions (rows)
    for idx in range(M-B):
      x_m_burnin = burnin_x_samples[:, idx:idx+1] # Shape (D,1)
      # (x_m_burnin - th) will broadcast to (D, 1).
      # We first sum only along D with .sum(0), then average over M-B.
      avg_grad_th += ((x_m_burnin - th).sum(0) / 5)

    th = th + delta_step * avg_grad_th /(M-B) # Update theta
    th_list.append(np.copy(th)) # Store new theta

  return th_list, x_values

And this is the gradient function specific to the problem.

In [ ]:
def log_p_grad_x(th, x, y_l, y_f):
  s = 1/(1+np.exp(- np.matmul(y_f, x)))
  return np.matmul((y_l-s).transpose(), y_f).transpose() - (x-th)/5


This is the SOUL function produced by Kuntz et al. for their *Particle algorithms for maximum likelihood training of latent variable models* (2023) article, that can be found [here](https://github.com/juankuntz/ParEM/blob/main/README.md).
We will use this as a 'control' to compare our results and the performance of our SOUL function.

In [ ]:
#@title SOUL control
def soul_ctrl(l, f, h, K, N, th, X):
    for k in range(K):
        # Run ULA chain:
        for n in range(N):
            Xkn = X[:, -1].reshape(D, 1)  # Extract current particle position.
            # Take a step:
            Xknp1 = (Xkn + h*grad_x(th[k], Xkn, l, f)
                          + np.sqrt(2*h)*np.random.normal(0, 1, (D, 1)))
            X = np.append(X, Xknp1, axis=1)  # Store new particle position.
        th = np.append(th, th[k] + h*ave_grad_th(th[k], X[:, -N:]))  # Update theta.
    return th, X

    # Auxiliary functions.

def ave_grad_th(th, x):
    """Returns theta-gradient of log density averaged over particle cloud."""
    return ((x-th).sum(0)).mean()/5

def grad_x(th, x, l, f):
    """Returns x-gradient of log density vectorized over particles."""
    s = 1/(1+np.exp(- np.matmul(f, x)))
    return np.matmul((l-s).transpose(), f).transpose() - (x-th)/5

# SOUL variants

As explained in the report, it is frequent that we can not derive the gradient of the prior p. Therefore, we wish to test and compare different gradient-free variants of the SOUL algorithm on this example. Here is our auxiliary function for this:

In [ ]:
def log_p(th, x, y_l, y_f):
    f_T_x = np.dot(y_f, x)
    log_lik = np.sum(y_l*f_T_x - np.log(1+np.exp(f_T_x)))
    log_prior = np.sum((x-th) ** 2) / 5.0
    return log_lik - log_prior

This is our first variant: soul_mh uses Metropolis-Hastings instead of ULA.

In [ ]:

#@title SOUL with Metropolis-Hastings.
def soul_mh(log_p, th0, x0_M, y_l, y_f, T, M, B, delta_step, proposal_std):
  """
  SOUL where the latent sampling is done via Metropolis-Hastings (MH) instead of ULA.

  Parameters:
  - log_p: Function returning log p(th, X, y_l, y_f). Returns a scalar float.
  - th0: Initial parameters
  - x0_M: Initial latent variables from the previous step
  - y_l, y_f: Observed data
  - T: Number of outer optimization steps
  - M: Number of MH steps
  - B: Burn-in steps
  - delta_step: Step size for theta update
  - proposal_std: Standard deviation of the Gaussian random walk proposal (replaces gamma_step)
  """
  th = np.copy(th0)
  x_t = np.copy(x0_M[:, 0:1]).reshape(D, 1)

  x_values = np.array(x0_M)
  th_list = [np.copy(th)]
  al=0

  for t in range(1, T + 1):

    # Metropolis-Hastings step
    for m in range(1, M + 1):
      z = np.random.normal(0.0, 1.0, x_t.shape)
      x_prop = x_t + proposal_std*z

      # The proposal is symmetri ie. q(x_t | x_prop) = q(x_prop | x_t), so the proposal ratio cancels out.
      log_alpha = min(0, log_p(th, x_prop, y_l, y_f) - log_p(th, x_t, y_l, y_f))

      # Accept or reject
      if np.log(np.random.uniform(0.0, 1.0)) < log_alpha:
          x_t = x_prop # Accept proposal
      # else: x_t remains the same

      # Store the current position (accpted or not)
      x_values = np.append(x_values, np.copy(x_t), axis=1)

    burnin_x_samples = x_values[:, -(M-B):] # Shape (D, M-B)

    # Compute average gradient with respect to theta over the kept samples
    avg_grad_th = np.zeros_like(th)
    for idx in range(M - B):
      x_m_burnin = burnin_x_samples[:, idx:idx+1] # Shape (D, 1)
      # Use the provided theta gradient function
      avg_grad_th += ((x_m_burnin - th).sum(0) / 5)

    # Update theta
    th = th + delta_step * (avg_grad_th / (M - B))
    th_list.append(np.copy(th))

  return th_list, x_values

In [ ]:
#@title SOUL MHSS
def soul_mh_ss(log_p, th0, x0_M, y_l, y_f, T, M, B, delta_step, proposal_std, lower_bounds, upper_bounds):
  """
  SOUL where the latent sampling is done via Metropolis-Hastings with Standardised Scaling (MH SS).

  Parameters:
  - log_p: Function returning log p(th, X, y_l, y_f). Returns a scalar float.
  - th0: Initial parameters
  - x0_M: Initial latent variables from the previous step
  - y_l, y_f: Observed data
  - T: Number of outer optimization steps
  - M: Number of MH steps
  - B: Burn-in steps
  - delta_step: Step size for theta update
  - proposal_std: Standard deviation of the Gaussian random walk proposal in the [0, 1] space (typically 0.05 to 0.2)
  - lower_bounds: numpy array of shape (D, 1) containing the lower prior limits for X
  - upper_bounds: numpy array of shape (D, 1) containing the upper prior limits for X
  """
  th = np.copy(th0)
  D = x0_M.shape[0]
  x_t = np.copy(x0_M[:, 0:1]).reshape(D, 1)

  bounds_range = upper_bounds - lower_bounds

  x_values = np.array(x0_M)
  th_list = [np.copy(th)]

  for t in range(1, T + 1):

    for m in range(1, M + 1):
      # Transform current position to [0, 1] standardised space
      x_t_std = (x_t - lower_bounds) / bounds_range

      z = np.random.normal(0.0, 1.0, x_t.shape)
      x_prop_std = x_t_std + proposal_std * z

      # proposals must stay within [0, 1]
      if np.any(x_prop_std < 0.0) or np.any(x_prop_std > 1.0):
        # Out of bounds proposal: target density is effectively 0, so we reject it.
        x_values = np.append(x_values, np.copy(x_t), axis=1)
        continue

      x_prop = x_prop_std * bounds_range + lower_bounds

      # Since the proposal is symmetric in the standardised space and we reject
      # out-of-bounds proposals, the proposal ratio q(x_t | x_prop) / q(x_prop | x_t) remains 1.
      log_alpha = min(0, log_p(th, x_prop, y_l, y_f) - log_p(th, x_t, y_l, y_f))

      # Accept or reject
      if np.log(np.random.uniform(0.0, 1.0)) < log_alpha:
        x_t = x_prop

      # Store the current position
      x_values = np.append(x_values, np.copy(x_t), axis=1)

    burnin_x_samples = x_values[:, -(M - B):]

    avg_grad_th = np.zeros_like(th)
    for idx in range(M - B):
      x_m_burnin = burnin_x_samples[:, idx:idx + 1]  # Shape (D, 1)
      avg_grad_th += ((x_m_burnin - th).sum(0) / 5)

    # Update theta
    th = th + delta_step * (avg_grad_th / (M - B))
    th_list.append(np.copy(th))

  return th_list, x_values

In [ ]:
#@title SOUL with PAIES algorithm and the stretch move
def soul_stretch(log_p, th0, x0_N, y_l, y_f, T, M, B, delta_step, a=2.0):
    """
    Stochastic Optimisation via Affine-Invariant Ensemble SOUL - stretch move.

    Parameters:
    - log_p: Function returning log p(th, X, y_l, y_f). Shape of X is (D, 1).
    - th0: Initial parameters
    - x0_N: Initial latent variables ensemble (S), shape (D, N) where N is number of walkers
    - y_l, y_f: Observed data
    - T: Number of outer optimization steps
    - M: Target chain length (number of ensemble update steps)
    - B: Burn-in steps
    - delta_step: Step size for theta update
    - a: Stretch move scale parameter (typically 2.0)
    """
    th = np.copy(th0)
    D, N = x0_N.shape
    half = N // 2

    # S is our active ensemble of N walkers, shape (D, N)
    S = np.copy(x0_N)

    # Collection C to store all states across the chain length M
    C = np.expand_dims(np.copy(S), axis=2)
    th_list = [th0]

    for t in range(1, T + 1):
        # Affine-Invariant Ensemble steps
        for m in range(1, M + 1):
            # Randomly shuffle and split S into two halves S1 and S2
            indices = np.random.permutation(N)
            S1_idx, S2_idx = indices[:half], indices[half:]

            # Update each half using the complementary half
            for ens_idx, comp_idx in [(S1_idx, S2_idx), (S2_idx, S1_idx)]:
                for i in ens_idx:
                    X_i = S[:, i:i+1] # shape D, 1

                    # Select a random walker X_j from the complementary ensemble
                    j = np.random.choice(comp_idx)
                    X_j = S[:, j:j+1]

                    # Propose new position using the stretch move rule R
                    u_z = np.random.uniform(0.0, 1.0)
                    z = (a + (1/a) - 2)*(u_z**2) + 2*u_z*(1-(1/a)) + (1/a)
                    X_i_new = X_j + z * (X_i - X_j)

                    log_alpha = (D - 1) * np.log(z) + log_p(th, X_i_new, y_l, y_f) - log_p(th, X_i, y_l, y_f)

                    # Accept or reject
                    if np.log(np.random.uniform(0.0, 1.0)) < log_alpha:
                        S[:, i:i+1] = X_i_new

            # Save new position (accepted or not)
            C = np.concatenate((C, np.expand_dims(np.copy(S), axis=2)), axis=2)

        burnin_ensemble_samples = C #[:, :, -(M - B):]  # Shape (D, N, M - B)
        # no burnin for a test

        # Reshape to treat all walker positions (post-burnin) as individual particles: shape (D, N * (M - B))
        flat_samples = burnin_ensemble_samples.reshape(D, -1)
        num_samples = flat_samples.shape[1]

        # Compute new theta using the averaged gradient
        avg_grad_th = np.zeros_like(th)
        for idx in range(num_samples):
            x_m_burnin = flat_samples[:, idx:idx+1]  # Shape (D, 1)
            avg_grad_th += ((x_m_burnin - th).sum(0) / 5)

        th = th + delta_step * avg_grad_th / num_samples
        th_list.append(np.copy(th))

    return th_list, C

In [ ]:

def soul_walk(log_p, th0, x0_N, y_l, y_f, T, M, B, delta_step):
    """
    Stochastic Optimisation via Affine-Invariant Ensemble SOUL - walk move.

    Parameters:
    - log_p: Function returning log p(th, X, y_l, y_f). Shape of X is (D, 1).
    - th0: Initial parameters
    - x0_N: Initial latent variables ensemble (S), shape (D, N) where N is number of walkers
    - y_l, y_f: Observed data
    - T: Number of outer optimization steps
    - M: Target chain length (number of ensemble update steps)
    - B: Burn-in steps
    - delta_step: Step size for theta update
    """
    th = np.copy(th0)
    D, N = x0_N.shape
    half = N // 2

    # S is our active ensemble of N walkers, shape (D, N)
    S = np.copy(x0_N)

    # Collection C to store all states across the chain length M
    C = np.expand_dims(np.copy(S), axis=2)
    th_list = [th0]

    for t in range(1, T + 1):
        # Affine-Invariant Ensemble steps
        for m in range(1, M + 1):
            # Randomly shuffle and split S into two halves S1 and S2
            indices = np.random.permutation(N)
            S1_idx, S2_idx = indices[:half], indices[half:]

            # Update each half using the complementary half
            for ens_idx, comp_idx in [(S1_idx, S2_idx), (S2_idx, S1_idx)]:

                # Extract the complementary ensemble
                S_comp = S[:, comp_idx]

                # Calculate the covariance matrix C of the complementary ensemble
                # ddof=0 enforces the 1/|S| normalization from Equation 13 in image_303966.png
                Cov = np.cov(S_comp, ddof=0)

                # Handle 1D case to ensure Cov remains a valid 2D matrix for the multivariate sampler
                if D == 1:
                    Cov = np.array([[Cov]])

                for i in ens_idx:
                    X_i = S[:, i:i+1] # shape D, 1

                    # Propose new position using the walk move rule: X_new = X_i + N(0, C)
                    # Sample step W from a multivariate normal distribution based on Cov
                    W = np.random.multivariate_normal(np.zeros(D), Cov).reshape(D, 1)
                    X_i_new = X_i + W

                    # Calculate acceptance log_alpha.
                    # The walk move is symmetric, so there is no (D - 1)*np.log(z) term.
                    log_alpha = log_p(th, X_i_new, y_l, y_f) - log_p(th, X_i, y_l, y_f)

                    # Accept or reject
                    if np.log(np.random.uniform(0.0, 1.0)) < log_alpha:
                        S[:, i:i+1] = X_i_new

            # Save new position (accepted or not)
            C = np.concatenate((C, np.expand_dims(np.copy(S), axis=2)), axis=2)

        burnin_ensemble_samples = C #[:, :, -(M - B):]  # Shape (D, N, M - B)
        # no burnin for a test

        # Reshape to treat all walker positions (post-burnin) as individual particles: shape (D, N * (M - B))
        flat_samples = burnin_ensemble_samples.reshape(D, -1)
        num_samples = flat_samples.shape[1]

        # Compute new theta using the averaged gradient
        avg_grad_th = np.zeros_like(th)
        for idx in range(num_samples):
            x_m_burnin = flat_samples[:, idx:idx+1]  # Shape (D, 1)
            avg_grad_th += ((x_m_burnin - th).sum(0) / 5)

        th = th + delta_step * avg_grad_th / num_samples
        th_list.append(np.copy(th))

    return th_list, C

# Figure 3a

Let's run the algorithm on 80/20 split of the data and plot the results.

In [ ]:
import numpy as np
# Split data into 80/20 training and testing sets:
from sklearn.model_selection import train_test_split
ftrain, ftest, ltrain, ltest = train_test_split(features, labels, test_size=0.2,
                                                random_state=0)

# Set approximation parameters as in our reference paper:
delta_step = 1e-2 # Step-size, same for ULA and Langevin in this case.
gamma_step = 1e-2
T = 400  # Number of steps.
M = 100  # Number of particles.
B = 0 # There is no burn-in in this example.
proposal_std = 0.25

# Initialize parameter estimates and particle cloud, all at zero:
D = features[0, :].size  # Dimension of latent space.
th0 = np.array([0.0])  # Parameter estimate, initialized as a (1,1) float array (scalar).
x0 = np.zeros((D, M))  # Particle cloud.

lower_bounds = np.min(x0, axis=1, keepdims=True)
upper_bounds = np.max(x0, axis=1, keepdims=True)

N = 10 # Number of walkers
x0_N = np.random.normal(0, 0.1, (D, N)) # Initialize x0_N with some slight noise

# Run algorithms:
th_soul, x_soul = soul(log_p_grad_x, th0, x0, ltrain, ftrain, T, M, B, delta_step, gamma_step)
#th_soul_mh, x_soul_mh = soul_mh(log_p, th0, x0, ltrain, ftrain, T, M, B, delta_step, proposal_std)
#th_soul_stretch, x_soul_stretch = soul_stretch(log_p, th0, x0_N, ltrain, ftrain, T, M, B, delta_step)
#th_soul_walk, x_soul_walk = soul_walk(log_p, th0, x0_N, ltrain, ftrain, T, M, B, delta_step)
th_soul_mh_ss, x_soul_mh_ss = soul_mh_ss(log_p, th0, x0_N, ltrain, ftrain, T, M, B, delta_step, proposal_std, lower_bounds, upper_bounds)

# Compare to the control function
#th_ctrl, x_ctrl = soul_ctrl(ltrain, ftrain, delta_step, T, M, np.array([[0]]), x0)

# Plot the results
plt.figure(figsize=(10, 6))

plt.plot(th_soul, label='SOUL')
#plt.plot(th_ctrl, label='SOUL (Control)')
#plt.plot(th_soul_mh, label='SOUL_MH')
#plt.plot(th_soul_stretch, label='SOUL_STRETCH')
#plt.plot(th_soul_walk, label='SOUL WALK')
plt.plot(th_soul_mh_ss, label='SOUL_MH_SS')

plt.xlim([-T/100, T])
plt.xlabel('Time Step')
plt.ylabel('Parameter Value')
plt.title('Evolution of SOUL Scalar Parameter theta')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
th_soul_mh_ss[-1]

In [ ]:
# Save the variables to a .npz file
np.savez('soul_variant_results.npz',
         th_soul_walk=th_soul_walk,
         x_soul_walk=x_soul_walk,
         th_soul_stretch=th_soul_stretch,
         x_soul_stretch=x_soul_stretch,
         th_soul_mh=th_soul_mh,
         x_soul_mh=x_soul_mh,
         th_soul=th_soul,
         x_soul=x_soul,
         th_ctrl=th_ctrl,
         x_ctrl=x_ctrl)

print("Variables saved to 'soul_variant_results.npz'")

In [ ]:
from google.colab import files

files.download('soul_variant_results.npz')

In [ ]:
from google.colab import files

# This will open a file picker dialog for you to select the .npz file
files.upload()

After uploading the file, we can then load the variables:

In [ ]:
loaded_data = np.load('soul_variant_results.npz')
th_soul_walk = loaded_data['th_soul_walk']
x_soul_walk = loaded_data['x_soul_walk']
th_soul_stretch = loaded_data['th_soul_stretch']
x_soul_stretch = loaded_data['x_soul_stretch']
th_soul_mh = loaded_data['th_soul_mh']
x_soul_mh = loaded_data['x_soul_mh']
th_soul = loaded_data['th_soul']
x_soul = loaded_data['x_soul']
th_ctrl = loaded_data['th_ctrl']
x_ctrl = loaded_data['x_ctrl']

print('Variables loaded successfully.')

We obtain the same allure as in the paper. Small differences between our SOUL function and the control function are due to the randomness within each run.

SOUL_MH - proposal_std tuned by hand for a healthy acceptance rate (around 30-40%) proposal_std=0.25 here (0.2 better than 0.3)

soul_stretch and soul_walk - very long to run (29min for both with N=10). the more walkers, the more precise. they are supposed to be better on more complex problems -->> code another example.
If they converge lower than soul control -> augment N or M, it's probably a gradient estimation error.

# Figure 3b

In [ ]:
# Extract final particle clouds X^{1:N}_K:
q_soul = x_soul[:, -M:]

# Generate and plot KDEs:
from scipy import stats  # stats to generate KDEs.
fig = plt.figure(figsize=(27,3), dpi= 100)

kde_r_list=[np.float64(2.58115862700272), np.float64(2.6487090518790377),
            np.float64(3.209504392076402), np.float64(2.3150004182323594),
            np.float64(2.2982599337403675), np.float64(3.050490617053107),
            np.float64(2.5301136977877774), np.float64(2.401086429683187),
            np.float64(2.498822615845701)]
kde_l_list=[np.float64(-0.015738804057094602), np.float64(-2.15085088023593),
            np.float64(-2.2304838689735194), np.float64(-0.2640190516743613),
            np.float64(-0.8586621768891025), np.float64(0.6472855588865127),
            np.float64(-0.019397450827125517), np.float64(-0.2697351136842945),
            np.float64(-0.28102164581791156)]

for i in range(D):
    # Generate KDEs for ith entry of the final particle cloud X^{1:N}_K:
    kde_min = kde_l_list[i]
    kde_max = kde_r_list[i]
    xaxis = np.linspace(kde_min, kde_max, num=100)

    kde_soul = stats.gaussian_kde(q_soul[i, :])(xaxis)

    # Plot KDEs:
    plt.subplot(1, D, i+1)
    plt.plot(xaxis, kde_soul, label='SOUL')
    plt.title('Feature ' + str(i+1))
    plt.ylim([0, 1.02*np.max([kde_soul])])
    plt.xlim([kde_min, kde_max])


handles, figlabels = plt.gca().get_legend_handles_labels()
fig.legend(handles, figlabels, ncol=4, bbox_to_anchor=(0.36,1.3),
           loc="upper center",fontsize=20)
plt.subplots_adjust(hspace=0.6)

Again, the allure is similar to that in the paper. However, since randomness plays a more important role here, we will average out 20 runs to make a better comparison.

# Average Figure 3b

In [ ]:
# Run the code 20 times and compute the averages

th_avg = np.zeros((T + 1, 1))
x_avg = np.zeros((D, M))


for i in range(30):
  th_soul_full, x_soul_full = soul(log_p_grad_x, th0, x0, ltrain, ftrain, T, M, B, delta_step, gamma_step)
  # Convert th_soul_list to a numpy array for addition
  th_avg += np.array(th_soul_full)
  # Only take the last M particles for x_avg
  x_avg += x_soul_full[:, -M:]

th_avg = th_avg/30
x_avg = x_avg/30

In [ ]:
th_avg_cntrl = np.zeros((T + 1, 1))
x_avg_cntrl = np.zeros((D, M))

for i in range(30):
  th_soul_cntrl, x_soul_cntrl = soul_ctrl(ltrain, ftrain, delta_step, T, M, np.array([[0]]), x0)
  # Convert th_soul_cntrl to a 2D numpy array with shape (T+1, 1) for addition
  th_avg_cntrl += th_soul_cntrl.reshape(-1, 1)
  # Only take the last M particles for x_avg_cntrl
  x_avg_cntrl += x_soul_cntrl[:, -M:]

th_avg_cntrl = th_avg_cntrl/30
x_avg_cntrl = x_avg_cntrl/30

In [ ]:
# Plot both averages

q_soul_avg = x_avg[:, -M:]
q_soul_avg_cntrl = x_avg_cntrl[:, -M:]

# Generate and plot KDEs:
from scipy import stats  # stats to generate KDEs.
import math # For ceil function

# Calculate number of columns for the subplots
num_cols = math.ceil(D / 2)

fig = plt.figure(figsize=(20, 8), dpi= 100) # Adjusted figure size for two rows

# These are based on previous plots.
# The exact values are arbitrary but allow us to compare easily the plots.
kde_r_list=[2.0, 1.0, 3.0, 4.0, 2.0, 4.0, 2.0, 2.0, 2.0]
kde_l_list=[0.0, -1.0, -1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]

for i in range(D):
    # Generate KDEs for ith entry of the final particle cloud X^{1:N}_K:
    kde_min = kde_l_list[i]
    kde_max = kde_r_list[i]
    xaxis = np.linspace(kde_min, kde_max, num=100)

    kde_soul_avg = stats.gaussian_kde(q_soul_avg[i, :])(xaxis)
    kde_soul_avg_cntrl = stats.gaussian_kde(q_soul_avg_cntrl[i, :])(xaxis)

    # Plot KDEs:
    plt.subplot(2, num_cols, i+1)
    plt.plot(xaxis, kde_soul_avg, label='my_version' )
    plt.plot(xaxis, kde_soul_avg_cntrl, label='control')
    plt.title('Feature ' + str(i+1), fontsize=22)
    plt.ylim([0, 1.2*np.max([kde_soul_avg, kde_soul_avg_cntrl])])
    plt.xlim([kde_min, kde_max])
    plt.xlabel('Value', fontsize=18)
    plt.ylabel('Density', fontsize=18)
    plt.xticks(fontsize=20)
    plt.yticks(fontsize=20)

handles, figlabels = plt.gca().get_legend_handles_labels()
fig.legend(handles, figlabels, ncol=4, bbox_to_anchor=(0.5, 1.05), # Adjusted legend position
           loc="upper center",fontsize=22)
plt.subplots_adjust(hspace=0.6)
plt.tight_layout(rect=[0, 0, 1, 0.95]) # Adjust layout to prevent title/legend overlap

# Figure 3c


In [ ]:
th0 = np.array([10.]) # Float
x0 = 10*np.ones((D, M))


In [ ]:
# Re-run algorithm:
th_soul, x_soul = soul(log_p_grad_x, th0, x0, ltrain, ftrain, T, M, B, delta_step, gamma_step)
th_ctrl, x_ctrl = soul_ctrl(ltrain, ftrain, delta_step, T, M, np.array([[10]]), x0)


# Plot parameter estimates:
plt.plot(th_soul, label='my_soul')
plt.plot(th_ctrl, label='control')
plt.xlabel('Time Step')
plt.ylabel('Parameter Value')
plt.title('Evolution of SOUL Scalar Parameter theta')
plt.grid(True)
plt.xlim([-T/100, T])
plt.legend(loc='upper right')

# Performance

We benchmark performance of the algorithms by running them several times. First, we examine their predictive performance, which we evaluate using two metrics: the test error and log pointwise predictive density.